# Chapter 5: Automated Evaluators
Code-based checks, an LLM-as-judge, testing the judge (TPR / TNR), the corrected success rate with a confidence interval, and group metrics.

## Run in Google Colab or locally
Upload this notebook alone and run cells from top to bottom. Setup installs missing packages;
all required sample data and helper functions are included below.
For Gemini calls, add `GEMINI_API_KEY` in Colab's key-shaped **Secrets** panel and enable notebook access,
or enter your key at the hidden prompt. Local runs use the single repository-root `.env`.
Never paste a key into a code cell. API calls use your own quota; notebook 04 needs no API key.
Students: **File → Save a copy in Drive** for your own copy and session.
Generated files live in the current session; download any results you want to keep before it ends.


In [ ]:
# Run once per fresh Colab session. Local users can run uv sync at the repo root.
import importlib.util
import subprocess
import sys

packages = {"google.genai": "google-genai", "dotenv": "python-dotenv",
            "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "rank_bm25": "rank-bm25"}
def available(module):
    try:
        return importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        return False
missing = [package for module, package in packages.items() if not available(module)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


In [ ]:
"""Small helpers shared by all notebooks (Gemini calls + data loading)."""
import json
import os
import time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Colab uses its own Secrets; local runs find the root .env.
try:
    from google.colab import userdata
except ImportError:
    userdata = None
if userdata is None:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / ".env").is_file():
            load_dotenv(directory / ".env", override=False)
            break

MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
EMBED_MODEL = os.getenv("GEMINI_EMBED_MODEL", "gemini-embedding-001")

_client = None


def client():
    """Create the Gemini client once and reuse it."""
    global _client
    if _client is None:
        key = os.getenv("GEMINI_API_KEY")
        if not key and userdata is not None:
            try:
                key = userdata.get("GEMINI_API_KEY")
            except Exception:
                pass
        if not key:
            from getpass import getpass
            key = getpass("Enter your own Gemini API key (hidden): ").strip()
        if not key:
            raise RuntimeError("GEMINI_API_KEY is missing. Add it to Colab Secrets or the root .env.")
        _client = genai.Client(api_key=key)
    return _client



def _generate(contents, system=None, temperature=0.0, as_json=False, model=None, retries=5):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json" if as_json else None,
    )
    for attempt in range(retries):
        try:
            resp = client().models.generate_content(model=model or MODEL, contents=contents, config=config)
            return json.loads(resp.text) if as_json else resp.text
        except Exception as e:  # rate limits (free tier) or temporary errors
            if attempt == retries - 1:
                raise
            wait = 10 * (attempt + 1)
            print(f"[retry {attempt + 1}] {type(e).__name__}: waiting {wait}s")
            time.sleep(wait)


def ask(prompt, system=None, temperature=0.0, as_json=False, model=None):
    """One prompt in, one answer out. as_json=True returns a Python dict."""
    return _generate(prompt, system, temperature, as_json, model)


def chat(history, system=None, temperature=0.0, model=None):
    """history = [{"role": "user" | "assistant", "content": "..."}, ...]"""
    contents = [
        types.Content(role="model" if m["role"] == "assistant" else "user",
                      parts=[types.Part(text=m["content"])])
        for m in history
    ]
    return _generate(contents, system, temperature, False, model)


def embed(texts):
    """Turn a list of texts into a numpy matrix (one row per text)."""
    r = client().models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in r.embeddings])


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# All sample prompts and data are embedded in this notebook.
def load_data():
    return {'ch5': {'judge': {'task': 'Check ONE thing only: is the tone of this real estate agent message appropriate for a LUXURY buyer?', 'pass_definition': "Formal, polite and professional wording. No slang, no pressure tactics, no jokes at the client's expense.", 'fail_definition': 'Uses slang, lots of exclamation marks, pressure tactics, or a casual or disrespectful tone.'}, 'pass_items': [{'text': 'Good afternoon, Mr. Hale. I have arranged a private viewing of the Bellevue residence on Friday at 11:00. Please let me know if another time would suit you better.', 'human_label': 'Pass'}, {'text': 'Dear Ms. Ortiz, the penthouse at Marina Tower has just become available. I would be delighted to share the floor plans at your convenience.', 'human_label': 'Pass'}, {'text': "Thank you for your patience. The architect's full specifications for the Highland estate are attached for your review.", 'human_label': 'Pass'}, {'text': 'It would be my pleasure to coordinate a discreet tour of the lakeside property this weekend.', 'human_label': 'Pass'}, {'text': 'Allow me to summarize the three properties that best match your criteria: a historic townhouse, a modern villa, and a waterfront estate.', 'human_label': 'Pass'}, {'text': 'Good morning. Your appointment with the interior designer has been confirmed for Tuesday at 2:00 p.m.', 'human_label': 'Pass'}, {'text': 'The property features a private wine cellar, a heated infinity pool, and 24-hour concierge service.', 'human_label': 'Pass'}, {'text': 'I appreciate your interest. Please find the comparative market analysis for the Ridgewood estate enclosed.', 'human_label': 'Pass'}, {'text': 'May I suggest an evening showing so you may enjoy the sunset views from the terrace?', 'human_label': 'Pass'}, {'text': 'Should you wish to proceed, I will prepare a formal offer for your consideration.', 'human_label': 'Pass'}, {'text': 'The seller has accepted your terms. Congratulations; I will coordinate the closing details with your attorney.', 'human_label': 'Pass'}, {'text': 'We have several off-market listings that may interest you. I would be glad to arrange a private presentation.', 'human_label': 'Pass'}, {'text': 'Certainly. The estate is 8,200 square feet on two acres, with a separate guest house.', 'human_label': 'Pass'}, {'text': 'Please accept my apologies for the delay. A revised itinerary for your visit is attached.', 'human_label': 'Pass'}, {'text': 'Welcome back, Mr. Hale. I trust your flight was pleasant. The car will be waiting at 9:00.', 'human_label': 'Pass'}], 'fail_items': [{'text': "Hey!! Got this awesome place, you'll LOVE it. Let's check it out ASAP!!!", 'human_label': 'Fail'}, {'text': "Yo, this house is a total steal, don't miss out lol.", 'human_label': 'Fail'}, {'text': "Hurry up and decide, other buyers are circling this one. Seriously, it won't last.", 'human_label': 'Fail'}, {'text': "Sup. Pool's sick, kitchen's dope. Wanna swing by tmrw?", 'human_label': 'Fail'}, {'text': "Dude, the price is crazy low for this area. Grab it before it's gone!!", 'human_label': 'Fail'}, {'text': "OMG you have to see this mansion!!! It's literally amazing!!!", 'human_label': 'Fail'}, {'text': "Whatever works for you, just lemme know. I'm kinda busy this week tbh.", 'human_label': 'Fail'}, {'text': "BTW the seller is being a pain about the price, but I'll deal with him.", 'human_label': 'Fail'}, {'text': "Lol that's a pricey one, you sure you can afford it?", 'human_label': 'Fail'}, {'text': 'Check this out: 5 bed, 4 bath, big yard, price 2.1M. Cheap, right?', 'human_label': 'Fail'}, {'text': "FYI the deadline is today. Don't blame me if you miss it.", 'human_label': 'Fail'}, {'text': 'Hey boss, new listings dropped. Peep these and hit me back.', 'human_label': 'Fail'}, {'text': "No worries, we'll find you something. The market is crazy right now, so who knows.", 'human_label': 'Fail'}, {'text': 'Honestly the neighbors are kinda loud but whatever, the house is great.', 'human_label': 'Fail'}, {'text': "Just sign here and we're done. Easy peasy.", 'human_label': 'Fail'}], 'unlabeled_texts': ['Good evening. I have shortlisted two estates that match your requirements and will send details shortly.', 'Yo, new listing just dropped, you gotta see it lol!!', 'Kindly find attached the inspection report for the Lakeview residence.', "Hurry, this one is going fast, don't wait!!!", 'It would be an honour to host you for a private tour on Saturday.', 'Sup, wanna check out the villa tmrw?', 'Thank you for your time today. I will follow up with the revised offer by noon.', 'Dude, this price is nuts, take it!', 'Please let me know whether Thursday morning suits you for the walkthrough.', 'Check it out, super cool house, easy deal.'], 'code_eval_outputs': ['{"sender": "Ali", "bullets": ["Meeting moved", "Send deck", "Book room"]}', '{"sender": "Sara", "bullets": ["Approve invoice", "Pay vendor"]}', 'Sender: Hina - logos uploaded', '{"sender": "Omar", "bullets": ["Budget approved", "Hire 2 engineers", "Start Monday"]}'], 'suggestions_diverse': ['Visit the lakeside cafe at sunrise', 'Try the pottery workshop downtown', 'Take the evening ferry to the island', 'Join the Saturday farmers market', 'Hike the old fort trail'], 'suggestions_repetitive': ['Visit the lakeside cafe at sunrise', 'Go to the lakeside cafe at sunrise', 'See the lakeside cafe early morning', 'Have coffee at the lakeside cafe at dawn', 'Stop by the cafe by the lake at sunrise'], 'retrieval_example': {'top5_is_relevant': [1, 0, 1, 1, 0], 'total_relevant_in_corpus': 10}, 'pass_at_k_attempts': [False, False, True, False, False]}}


In [ ]:
import json, random
import numpy as np

data = load_data()["ch5"]

## 1. Code-based evaluators
Fast, free, same answer every time. Use when the rule is objective.

In [ ]:
def valid_json(text):
    try:
        json.loads(text); return True
    except json.JSONDecodeError:
        return False

def exactly_3_bullets(text):
    try:
        return len(json.loads(text)["bullets"]) == 3
    except Exception:
        return False

for out in data["code_eval_outputs"]:
    print(f"valid_json={valid_json(out)!s:5}  3_bullets={exactly_3_bullets(out)!s:5}  {out[:50]}")

## 2. Split the labeled data: train / dev / test
- **train**: examples you may put inside the judge prompt
- **dev**: used to improve the prompt
- **test**: used once at the end (never look before)

In [ ]:
rng = random.Random(42)
pass_items, fail_items = data["pass_items"][:], data["fail_items"][:]
rng.shuffle(pass_items); rng.shuffle(fail_items)

train = pass_items[:2] + fail_items[:2]
dev   = pass_items[2:8] + fail_items[2:9]
test  = pass_items[8:] + fail_items[9:]

print("train:", len(train), "| dev:", len(dev), "| test:", len(test))

## 3. Build the LLM judge
Four parts: **one task**, Pass/Fail definitions, few-shot examples (from *train* only), JSON output.

In [ ]:
J = data["judge"]

def build_judge_prompt(text):
    shots = "\n".join(f'Message: "{x["text"]}"\nAnswer: {x["human_label"]}\n' for x in train)
    return f"""{J["task"]}

PASS means: {J["pass_definition"]}
FAIL means: {J["fail_definition"]}

Examples:
{shots}
Now judge this message:
Message: "{text}"

Reply as JSON: {{"reasoning": "one short sentence", "answer": "Pass" or "Fail"}}"""

def judge(text):
    out = ask(build_judge_prompt(text), as_json=True)
    return 1 if out["answer"].strip().lower() == "pass" else 0     # 1 = Pass, 0 = Fail

print(build_judge_prompt("Hey dude, nice house lol!"))
print("\nJudge says (1=Pass, 0=Fail):", judge("Hey dude, nice house lol!"))

## 4. Test the judge on the dev set: TPR and TNR
- **TPR**: of the real Passes, how many did the judge call Pass?
- **TNR**: of the real Fails, how many did the judge call Fail?

In [ ]:
def to_int(label):
    return 1 if label == "Pass" else 0

def tpr_tnr(human, preds):
    human, preds = np.array(human), np.array(preds)
    tpr = ((human == 1) & (preds == 1)).sum() / max((human == 1).sum(), 1)
    tnr = ((human == 0) & (preds == 0)).sum() / max((human == 0).sum(), 1)
    return tpr, tnr

dev_human = [to_int(x["human_label"]) for x in dev]
dev_preds = [judge(x["text"]) for x in dev]

tpr, tnr = tpr_tnr(dev_human, dev_preds)
print(f"DEV  TPR={tpr:.0%}  TNR={tnr:.0%}   (goal: both above ~90%)")

In [ ]:
# Look at the mistakes -> improve the prompt / definitions -> run again
for x, h, p in zip(dev, dev_human, dev_preds):
    if h != p:
        print(f"human={h} judge={p} | {x['text']}")

## 5. Final score on the test set (run once)

In [ ]:
test_human = [to_int(x["human_label"]) for x in test]
test_preds = [judge(x["text"]) for x in test]
tpr, tnr = tpr_tnr(test_human, test_preds)
print(f"TEST TPR={tpr:.0%}  TNR={tnr:.0%}")

## 6. True success rate + 95% confidence interval
The judge makes mistakes, so we **correct** its Pass rate:

`theta = (p_obs + TNR - 1) / (TPR + TNR - 1)`

The **bootstrap** re-draws the data many times to see how much the answer moves.

In [ ]:
def estimate_success_rate(test_labels, test_preds, unlabeled_preds, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    tl, tp, up = map(np.array, (test_labels, test_preds, unlabeled_preds))

    def theta_hat(tl, tp, up):
        tpr = ((tl == 1) & (tp == 1)).sum() / max((tl == 1).sum(), 1)
        tnr = ((tl == 0) & (tp == 0)).sum() / max((tl == 0).sum(), 1)
        if tpr + tnr - 1 <= 1e-9:
            return np.nan                                 # judge is no better than a coin flip
        return float(np.clip((up.mean() + tnr - 1) / (tpr + tnr - 1), 0, 1))

    theta = theta_hat(tl, tp, up)
    boots = []
    for _ in range(B):
        i = rng.integers(0, len(tl), len(tl))             # resample WITH repeats
        j = rng.integers(0, len(up), len(up))
        boots.append(theta_hat(tl[i], tp[i], up[j]))
    boots = [b for b in boots if not np.isnan(b)]
    if not boots:                                         # judge too weak to correct
        return theta, np.nan, np.nan
    L, U = np.percentile(boots, [2.5, 97.5])
    return theta, L, U

In [ ]:
unlabeled_preds = [judge(t) for t in data["unlabeled_texts"]]
print("Raw judge pass rate (biased):", np.mean(unlabeled_preds))

theta, L, U = estimate_success_rate(test_human, test_preds, unlabeled_preds, B=2000)
print(f"Corrected success rate: {theta:.2f}   95% CI [{L:.2f}, {U:.2f}]")

target = 0.85
print("Ship?", "yes, lower bound passes the target" if L >= target else "not sure: lower bound is below the target")

## 7. Metrics for several outputs at once

In [ ]:
# Pass@k : at least one of k attempts is correct
attempts = data["pass_at_k_attempts"]
print("Pass@5:", any(attempts))

# Precision@k and Recall@k
ex = data["retrieval_example"]
top5 = ex["top5_is_relevant"]
print("Precision@5:", sum(top5) / len(top5))
print("Recall@5   :", sum(top5) / ex["total_relevant_in_corpus"])

In [ ]:
# Average pairwise similarity: lower = more diverse suggestions
def avg_pairwise_similarity(texts):
    v = embed(texts)
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    sim = v @ v.T
    return sim[np.triu_indices_from(sim, k=1)].mean()

print("Diverse    :", round(avg_pairwise_similarity(data["suggestions_diverse"]), 2))
print("Repetitive :", round(avg_pairwise_similarity(data["suggestions_repetitive"]), 2))